# Implied Volatility Surface Forecasting - Colab Training Notebook

This notebook is fully self-contained and ready to run on Google Colab (with a **T4 GPU runtime** enabled). It will:
1. Install necessary dependencies (`arch`, `mlflow`, `yfinance`, etc.).
2. Seed every RNG (Python/NumPy/PyTorch) for reproducible runs.
3. Define the deep learning architectures (`ConvLSTM`, `LSTM`, `Transformer`, and the HAR-RV + LSTM `Hybrid`).
4. Define the custom `SmoothnessRegularizedLoss` incorporating strike/expiry spatial penalties and no-arbitrage terms.
5. Implement all econometric benchmarks (`Random Walk`, `Historical Mean`, `Exponential Smoothing`, `GARCH`, `HAR-RV`).
6. Evolve synthetic surfaces, run the comparative training loops, and output the final results table.
7. Save each trained model's weights as `checkpoint_<name>.pt` + a `.pt.json` metadata sidecar — the same
   state_dict + sidecar format `models/serialization.py` expects locally, so the downloaded files can be
   dropped straight into `models/` without any conversion step.

### Enable GPU in Colab:
Go to **Runtime** -> **Change runtime type** -> select **T4 GPU**.

### After running:
Download every `checkpoint_<name>.pt` and matching `checkpoint_<name>.pt.json` file and place both into
`models/` on your machine. To update the active served model (`models/checkpoint.pt`), copy whichever
model's pair matches `configs/base_config.yaml`'s `model.name` (default: `hybrid`) to `models/checkpoint.pt`
and `models/checkpoint.pt.json`.

In [ ]:
# 1. Install required libraries
!pip install arch mlflow yfinance pyyaml scipy scikit-learn pandas numpy torch

In [ ]:
!nvidia-smi
import torch

print("CUDA Available:", torch.cuda.is_available())
print("GPU Name:", torch.cuda.get_device_name(0))
print("Current Device:", torch.cuda.current_device())

In [ ]:
# 2. Imports
import os
import math
import json
import random
import yaml
import logging
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.linear_model import LinearRegression
from datetime import datetime, timedelta

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

In [ ]:
# Seed every RNG up front so this run (data generation, model init, training) is reproducible.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print(f"All RNGs seeded with SEED={SEED} for reproducibility.")

## Define Neural Networks

In [ ]:
# 3. Deep Learning Architectures

class StackedLSTM(nn.Module):
    def __init__(self, grid_size=(7, 7), hidden_dim=256, num_layers=3, horizon=1):
        super().__init__()
        self.M, self.N = grid_size
        self.input_dim = self.M * self.N
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.horizon = horizon
        self.lstm = nn.LSTM(input_size=self.input_dim, hidden_size=self.hidden_dim, num_layers=self.num_layers, batch_first=True)
        self.decoder = nn.Sequential(
            nn.Linear(self.hidden_dim, 512),
            nn.ReLU(),
            nn.Linear(512, self.horizon * self.input_dim)
        )
    def forward(self, x):
        B, L, C, M, N = x.shape
        x_flat = x.view(B, L, -1)
        out, _ = self.lstm(x_flat)
        last_hidden = out[:, -1, :]
        decoded = self.decoder(last_hidden)
        return decoded.view(B, self.horizon, self.M, self.N)


class ConvLSTMCell(nn.Module):
    def __init__(self, in_channels, hidden_dim, kernel_size=3, bias=True):
        super().__init__()
        self.in_channels = in_channels
        self.hidden_dim = hidden_dim
        self.kernel_size = kernel_size
        self.padding = kernel_size // 2
        self.conv = nn.Conv2d(
            in_channels=self.in_channels + self.hidden_dim,
            out_channels=4 * self.hidden_dim,
            kernel_size=self.kernel_size,
            padding=self.padding,
            bias=bias
        )
    def forward(self, x, cur_state):
        h_cur, c_cur = cur_state
        combined = torch.cat([x, h_cur], dim=1)
        combined_conv = self.conv(combined)
        cc_i, cc_f, cc_o, cc_g = torch.split(combined_conv, self.hidden_dim, dim=1)
        i = torch.sigmoid(cc_i)
        f = torch.sigmoid(cc_f)
        o = torch.sigmoid(cc_o)
        g = torch.tanh(cc_g)
        c_next = f * c_cur + i * g
        h_next = o * torch.tanh(c_next)
        return h_next, c_next
    def init_hidden(self, batch_size, image_size, device):
        height, width = image_size
        return (
            torch.zeros(batch_size, self.hidden_dim, height, width, device=device),
            torch.zeros(batch_size, self.hidden_dim, height, width, device=device)
        )


class ConvLSTM(nn.Module):
    def __init__(self, in_channels=1, hidden_dims=[32, 64, 64], kernel_size=3, num_layers=3, horizon=1):
        super().__init__()
        self.in_channels = in_channels
        self.hidden_dims = hidden_dims
        self.num_layers = num_layers
        self.horizon = horizon
        cell_list = []
        for i in range(self.num_layers):
            cur_in = self.in_channels if i == 0 else self.hidden_dims[i - 1]
            cell_list.append(ConvLSTMCell(in_channels=cur_in, hidden_dim=self.hidden_dims[i], kernel_size=kernel_size))
        self.cell_list = nn.ModuleList(cell_list)
        bn_list = []
        for i in range(self.num_layers - 1):
            bn_list.append(nn.BatchNorm3d(self.hidden_dims[i]))
        self.bn_list = nn.ModuleList(bn_list)
        self.decoder = nn.Sequential(
            nn.Conv2d(self.hidden_dims[-1], 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, self.horizon, kernel_size=1, padding=0)
        )
    def forward(self, x):
        B, L, C, M, N = x.shape
        device = x.device
        current_input = x
        for i in range(self.num_layers):
            cell = self.cell_list[i]
            h, c = cell.init_hidden(B, (M, N), device)
            outputs = []
            for t in range(L):
                h, c = cell(current_input[:, t, :, :, :], (h, c))
                outputs.append(h)
            outputs = torch.stack(outputs, dim=1)
            if i < self.num_layers - 1:
                outputs = outputs.permute(0, 2, 1, 3, 4)
                outputs = self.bn_list[i](outputs)
                outputs = outputs.permute(0, 2, 1, 3, 4)
            current_input = outputs
        last_hidden = current_input[:, -1, :, :, :]
        return self.decoder(last_hidden)


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]


class TransformerEncoderModel(nn.Module):
    def __init__(self, grid_size=(7, 7), d_model=128, nhead=4, num_layers=3, dim_feedforward=256, dropout=0.1, horizon=1):
        super().__init__()
        self.M, self.N = grid_size
        self.input_dim = self.M * self.N
        self.d_model = d_model
        self.horizon = horizon
        self.embedding = nn.Linear(self.input_dim, self.d_model)
        self.pos_encoder = PositionalEncoding(d_model=self.d_model)
        encoder_layer = nn.TransformerEncoderLayer(d_model=self.d_model, nhead=nhead, dim_feedforward=dim_feedforward, dropout=dropout, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.decoder = nn.Sequential(
            nn.Linear(self.d_model, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, self.horizon * self.input_dim)
        )
    def forward(self, x):
        B, L, C, M, N = x.shape
        x_flat = x.view(B, L, -1)
        embedded = self.embedding(x_flat)
        encoded = self.pos_encoder(embedded)
        out = self.transformer_encoder(encoded)
        last_step = out[:, -1, :]
        decoded = self.decoder(last_step)
        return decoded.view(B, self.horizon, self.M, self.N)


class HARRVLSTMHybrid(nn.Module):
    """
    HAR-RV + Gated LSTM Hybrid Model:
        Final Forecast (norm) = HAR-RV Base (norm) + sigmoid(g) * LSTM Residual (norm)
    The learnable scalar gate g is initialized to -3 so that sigmoid(g) ~= 0.05 at the
    start of training: the HAR-RV base dominates early on, and the LSTM residual only
    contributes once it has learned useful structure.
    """
    def __init__(self, grid_size=(7, 7), hidden_dim=64, num_layers=3, horizon=1):
        super().__init__()
        self.M, self.N = grid_size
        self.horizon = horizon

        self.register_buffer("har_coefs", torch.zeros(self.M, self.N, self.horizon, 3))
        self.register_buffer("har_intercepts", torch.zeros(self.M, self.N, self.horizon))

        self.lstm = StackedLSTM(grid_size=grid_size, hidden_dim=hidden_dim, num_layers=num_layers, horizon=horizon)
        self.gate_logit = nn.Parameter(torch.tensor(-3.0))
        self.last_residual = None

    def fit_har(self, X: np.ndarray, y: np.ndarray):
        """Fits HAR-RV coefficients per grid cell from raw training sequences.
        X: (N_samples, L, 1, M, N), y: (N_samples, h, M, N)"""
        N_samples, L, _, M, N = X.shape
        if L < 20:
            raise ValueError(f"HAR-RV requires lookback window L >= 20. Got L={L}")

        coefs_np = np.zeros((self.M, self.N, self.horizon, 3))
        intercepts_np = np.zeros((self.M, self.N, self.horizon))

        for i in range(self.M):
            for j in range(self.N):
                features, targets = [], []
                for s in range(N_samples):
                    ts = X[s, :, 0, i, j]
                    val_d = ts[-1]
                    val_w = np.mean(ts[-5:])
                    val_m = np.mean(ts[-20:])
                    features.append([val_d, val_w, val_m])
                    targets.append(y[s, :, i, j])

                reg = LinearRegression()
                reg.fit(np.array(features), np.array(targets))

                coef = reg.coef_
                if self.horizon == 1:
                    coef = coef[np.newaxis, :]
                coefs_np[i, j] = coef
                intercepts_np[i, j] = reg.intercept_

        self.har_coefs.copy_(torch.from_numpy(coefs_np).float())
        self.har_intercepts.copy_(torch.from_numpy(intercepts_np).float())

    def predict_har_tensor(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, L, 1, M, N) unnormalized"""
        B, L, _, M, N = x.shape
        device = x.device

        val_d = x[:, -1, 0, :, :]
        val_w = torch.mean(x[:, -5:, 0, :, :], dim=1)
        val_m = torch.mean(x[:, -20:, 0, :, :], dim=1)
        features = torch.stack([val_d, val_w, val_m], dim=-1)

        har_pred = torch.zeros(B, self.horizon, M, N, device=device)
        for i in range(M):
            for j in range(N):
                proj = torch.matmul(features[:, i, j], self.har_coefs[i, j].t())
                har_pred[:, :, i, j] = proj + self.har_intercepts[i, j]

        return torch.clamp(har_pred, 0.01, 5.0)

    @property
    def gate_value(self) -> float:
        return torch.sigmoid(self.gate_logit).item()

    def forward(self, x):
        """x: standardized sequences (B, L, 1, M, N). Returns forecast in normalized space."""
        train_mean = getattr(self, "train_mean", 0.0)
        train_std = getattr(self, "train_std", 1.0)
        x_unnorm = x * train_std + train_mean

        har_pred_unnorm = self.predict_har_tensor(x_unnorm)
        har_pred_norm = (har_pred_unnorm - train_mean) / train_std

        residual_pred_norm = self.lstm(x)
        self.last_residual = residual_pred_norm

        gate = torch.sigmoid(self.gate_logit)
        return har_pred_norm + gate * residual_pred_norm

## Define Custom Loss Function

In [ ]:
# 4. Smoothness-Regularized Loss Function
# Grid convention: dim -1 = kappa (strike/moneyness), dim -2 = tau (expiry).

class SmoothnessRegularizedLoss(nn.Module):
    def __init__(self, lambda_strike=0.01, lambda_expiry=0.01, lambda_calendar=0.0,
                 lambda_butterfly=0.0, lambda_residual_mean=0.0, grid_taus=None):
        super().__init__()
        self.lambda_strike = lambda_strike
        self.lambda_expiry = lambda_expiry
        self.lambda_calendar = lambda_calendar
        self.lambda_butterfly = lambda_butterfly
        self.lambda_residual_mean = lambda_residual_mean

        taus = grid_taus if grid_taus is not None else np.array([1/52, 2/52, 1/12, 2/12, 3/12, 6/12, 1.0])
        self.register_buffer("grid_taus_tensor", torch.from_numpy(taus).float())

    def forward(self, y_pred, y_true, residual=None):
        mse_loss = F.mse_loss(y_pred, y_true)

        # Strike smoothness (dim=-1, kappas) with reflection padding
        padded_strike = F.pad(y_pred, pad=(1, 1, 0, 0), mode="reflect")
        strike_d2 = padded_strike[:, :, :, 2:] - 2 * padded_strike[:, :, :, 1:-1] + padded_strike[:, :, :, :-2]
        strike_loss = torch.mean(strike_d2 ** 2)

        # Expiry smoothness (dim=-2, taus) with reflection padding
        padded_expiry = F.pad(y_pred, pad=(0, 0, 1, 1), mode="reflect")
        expiry_d2 = padded_expiry[:, :, 2:, :] - 2 * padded_expiry[:, :, 1:-1, :] + padded_expiry[:, :, :-2, :]
        expiry_loss = torch.mean(expiry_d2 ** 2)

        # Calendar spread arbitrage penalty (along dim=-2, taus): total variance w = IV^2 * tau
        # must be non-decreasing in tau.
        calendar_loss = torch.tensor(0.0, device=y_pred.device)
        if self.lambda_calendar > 0:
            taus = self.grid_taus_tensor.view(1, 1, -1, 1)
            w = (y_pred ** 2) * taus
            w_diff = w[:, :, :-1, :] - w[:, :, 1:, :]
            calendar_loss = torch.mean(torch.relu(w_diff) ** 2)

        # Butterfly spread smile convexity penalty (along dim=-1, kappas): smile must be convex.
        butterfly_loss = torch.tensor(0.0, device=y_pred.device)
        if self.lambda_butterfly > 0:
            d2 = y_pred[:, :, :, 2:] - 2 * y_pred[:, :, :, 1:-1] + y_pred[:, :, :, :-2]
            butterfly_loss = torch.mean(torch.relu(-d2) ** 2)

        # Residual zero-mean penalty for the Hybrid model's LSTM residual sub-network.
        residual_mean_loss = torch.tensor(0.0, device=y_pred.device)
        if self.lambda_residual_mean > 0 and residual is not None:
            residual_mean_loss = residual.mean() ** 2

        total_loss = (
            mse_loss
            + self.lambda_strike * strike_loss
            + self.lambda_expiry * expiry_loss
            + self.lambda_calendar * calendar_loss
            + self.lambda_butterfly * butterfly_loss
            + self.lambda_residual_mean * residual_mean_loss
        )
        return total_loss, {
            "mse_loss": mse_loss.item(),
            "strike_loss": strike_loss.item(),
            "expiry_loss": expiry_loss.item(),
            "calendar_loss": calendar_loss.item(),
            "butterfly_loss": butterfly_loss.item(),
            "residual_mean_loss": residual_mean_loss.item(),
            "total_loss": total_loss.item()
        }

## Define Baselines

In [ ]:
# 5. Econometric Baselines

class NaiveRandomWalk:
    """Predicts that future surfaces will remain identical to the last observed surface."""
    def __init__(self, horizon=1):
        self.horizon = horizon
    def fit(self, X, y):
        pass
    def predict(self, X):
        last_step = X[:, -1, 0, :, :]
        return np.repeat(last_step[:, np.newaxis, :, :], self.horizon, axis=1)


class HistoricalMean:
    """Predicts that future surfaces will be the mean of the lookback window."""
    def __init__(self, horizon=1):
        self.horizon = horizon
    def fit(self, X, y):
        pass
    def predict(self, X):
        mean_surface = np.mean(X[:, :, 0, :, :], axis=1)
        return np.repeat(mean_surface[:, np.newaxis, :, :], self.horizon, axis=1)


class ExponentialSmoothing:
    """Single Exponential Smoothing: S_t = alpha * Y_t + (1 - alpha) * S_{t-1}, fitted per grid cell."""
    def __init__(self, horizon=1, alpha=0.3):
        self.horizon = horizon
        self.alpha = alpha
    def fit(self, X, y):
        pass
    def predict(self, X):
        B, L, _, M, N = X.shape
        predictions = np.zeros((B, self.horizon, M, N))
        for b in range(B):
            for i in range(M):
                for j in range(N):
                    ts = X[b, :, 0, i, j]
                    s = ts[0]
                    for t in range(1, L):
                        s = self.alpha * ts[t] + (1.0 - self.alpha) * s
                    predictions[b, :, i, j] = s
        return predictions


class GARCHModel:
    """GARCH(1,1) per grid cell via the 'arch' package. Falls back to Historical Mean if unavailable."""
    def __init__(self, horizon=1):
        self.horizon = horizon
        self._has_arch = False
        try:
            import arch
            self._has_arch = True
        except ImportError:
            print("The 'arch' package is not installed. GARCH model will fall back to Historical Mean.")
    def fit(self, X, y):
        pass
    def predict(self, X):
        B, L, _, M, N = X.shape
        predictions = np.zeros((B, self.horizon, M, N))
        if not self._has_arch:
            return HistoricalMean(horizon=self.horizon).predict(X)
        from arch import arch_model
        for b in range(B):
            for i in range(M):
                for j in range(N):
                    ts = X[b, :, 0, i, j]
                    try:
                        scale = 100.0
                        scaled_ts = ts * scale
                        model = arch_model(scaled_ts, mean="Constant", vol="GARCH", p=1, q=1, dist="normal")
                        res = model.fit(disp="off", show_warning=False)
                        forecasts = res.forecast(horizon=self.horizon, reindex=False)
                        cond_var = forecasts.variance.values[0]
                        forecast_iv = np.sqrt(np.clip(cond_var, 1e-6, None)) / scale
                        predictions[b, :, i, j] = forecast_iv
                    except Exception:
                        predictions[b, :, i, j] = np.mean(ts)
        return predictions


class HARRVModel:
    """HAR-RV: sigma_{t+1} = b0 + b_d*sigma_t^(1) + b_w*sigma_t^(5) + b_m*sigma_t^(20), fit per grid cell."""
    def __init__(self, horizon=1):
        self.horizon = horizon
        self.regressors = {}

    def fit(self, X, y):
        N_samples, L, _, M, N = X.shape
        if L < 20:
            raise ValueError(f"HAR-RV requires lookback window L >= 20. Got L={L}")
        for i in range(M):
            for j in range(N):
                features, targets = [], []
                for s in range(N_samples):
                    ts = X[s, :, 0, i, j]
                    val_d, val_w, val_m = ts[-1], np.mean(ts[-5:]), np.mean(ts[-20:])
                    features.append([val_d, val_w, val_m])
                    targets.append(y[s, :, i, j])
                reg = LinearRegression()
                reg.fit(np.array(features), np.array(targets))
                self.regressors[(i, j)] = reg

    def predict(self, X):
        B, L, _, M, N = X.shape
        predictions = np.zeros((B, self.horizon, M, N))
        for i in range(M):
            for j in range(N):
                features = []
                for b in range(B):
                    ts = X[b, :, 0, i, j]
                    features.append([ts[-1], np.mean(ts[-5:]), np.mean(ts[-20:])])
                reg = self.regressors.get((i, j))
                if reg is not None:
                    pred = reg.predict(np.array(features))
                    if self.horizon == 1:
                        pred = pred[:, np.newaxis]
                    predictions[:, :, i, j] = pred
                else:
                    last_step = X[:, -1, 0, i, j]
                    predictions[:, :, i, j] = np.repeat(last_step[:, np.newaxis], self.horizon, axis=1)
        return np.clip(predictions, 0.01, 5.0)

## Synthetic Data Generation & Sequencing

In [ ]:
# 6. Data Generator & Sequence builders

GRID_KAPPAS = np.array([-0.30, -0.20, -0.10, 0.00, 0.10, 0.20, 0.30])
GRID_TAUS = np.array([1/52, 2/52, 1/12, 2/12, 3/12, 6/12, 1.0])

def generate_synthetic_dataset(num_days=300, seed=None):
    """
    Generate a highly realistic synthetic volatility surface time-series.
    Incorporates Markov regime shifts (Calm vs. Crisis), Poisson jumps,
    leverage effects (skew-level coupling), and GARCH-like volatility clustering.

    Pass an explicit `seed` for reproducible output across Colab runs.
    """
    rng = np.random.default_rng(seed)

    level = 0.18
    skew = 0.05
    term = 0.03
    regime = 0  # 0: Calm, 1: Stressed/Crisis
    jump_effect = 0.0

    surfaces = []
    cond_vol = 0.015

    for day in range(num_days):
        if regime == 0:
            if rng.random() < 0.04:
                regime = 1
        else:
            if rng.random() < 0.12:
                regime = 0

        if regime == 0:
            mu_l, mu_s, mu_t = 0.16, 0.05, 0.03
            phi_l, phi_s, phi_t = 0.95, 0.90, 0.93
            sigma_l_base, sigma_s, sigma_t = 0.010, 0.004, 0.003
        else:
            mu_l, mu_s, mu_t = 0.35, 0.14, -0.02
            phi_l, phi_s, phi_t = 0.97, 0.92, 0.95
            sigma_l_base, sigma_s, sigma_t = 0.025, 0.009, 0.006

        cond_vol = 0.6 * cond_vol + 0.3 * (level - mu_l)**2 + 0.1 * sigma_l_base**2
        cond_vol = np.clip(np.sqrt(cond_vol), 0.005, 0.04)

        shock_l = rng.normal(0, cond_vol)
        shock_s = 0.4 * shock_l + rng.normal(0, sigma_s)
        shock_t = -0.2 * shock_l + rng.normal(0, sigma_t)

        jump = 0.0
        if rng.random() < 0.03:
            jump = rng.exponential(0.15)
            skew += 0.08

        jump_effect = 0.8 * jump_effect + jump
        level = mu_l + phi_l * (level - mu_l) + shock_l + jump
        skew = mu_s + phi_s * (skew - mu_s) + shock_s
        term = mu_t + phi_t * (term - mu_t) + shock_t

        effective_level = level + jump_effect
        effective_level = np.clip(effective_level, 0.08, 0.70)
        skew = np.clip(skew, 0.01, 0.25)
        term = np.clip(term, -0.06, 0.14)

        grid_iv = np.zeros((len(GRID_TAUS), len(GRID_KAPPAS)))
        for i, tau in enumerate(GRID_TAUS):
            for j, kappa in enumerate(GRID_KAPPAS):
                smile_curvature = 0.16 if regime == 1 else 0.10
                iv = effective_level - skew * kappa + smile_curvature * kappa**2 + term * np.log(tau / 0.25)
                grid_iv[i, j] = iv + rng.normal(0, 0.001)
        surfaces.append(np.clip(grid_iv, 0.02, 5.0))
    return np.stack(surfaces, axis=0)

def create_sequences(data, lookback, horizon):
    T, M, N = data.shape
    num_samples = T - lookback - horizon + 1
    X = np.zeros((num_samples, lookback, 1, M, N), dtype=np.float32)
    y = np.zeros((num_samples, horizon, M, N), dtype=np.float32)
    for i in range(num_samples):
        X[i, :, 0, :, :] = data[i : i + lookback]
        y[i, :, :, :] = data[i + lookback : i + lookback + horizon]
    return X, y

class SurfaceDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).float()
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

## Training Script & Evaluation

In [ ]:
# 7. Train Model Runner

def get_region_masks(kappas, taus):
    grid_k, grid_t = np.meshgrid(kappas, taus)
    return {
        "Overall": np.ones_like(grid_k, dtype=bool),
        "ATM": np.abs(grid_k) < 0.05,
        "OTM Puts": grid_k <= -0.10,
        "OTM Calls": grid_k >= 0.10,
        "Deep Wings": np.abs(grid_k) >= 0.20,
        "Short-dated": grid_t < 1/12,
        "Long-dated": grid_t > 6/12
    }

def calculate_metrics(y_pred, y_true):
    # Grid convention: dim -1 = kappa (strike), dim -2 = tau (expiry) -- matches training/train.py.
    rmse = np.sqrt(np.mean((y_pred - y_true) ** 2))
    mae = np.mean(np.abs(y_pred - y_true))
    mape = np.mean(np.abs(y_pred - y_true) / np.clip(y_true, 1e-5, None)) * 100.0
    tv_strike = np.mean(np.abs(y_pred[:, :, :, 1:] - y_pred[:, :, :, :-1]))
    tv_expiry = np.mean(np.abs(y_pred[:, :, 1:, :] - y_pred[:, :, :-1, :]))
    w_pred = (y_pred ** 2) * GRID_TAUS.reshape(1, 1, -1, 1)
    av_violations = np.sum(w_pred[:, :, :-1, :] > w_pred[:, :, 1:, :])
    av_ratio = av_violations / (y_pred.shape[0] * y_pred.shape[1] * (y_pred.shape[2] - 1) * y_pred.shape[3])
    return {
        "RMSE": rmse,
        "MAE": mae,
        "MAPE": mape,
        "TV": tv_strike + tv_expiry,
        "AV": av_ratio
    }

# --- Checkpoint serialization: state_dict + JSON sidecar, matching models/serialization.py ---
# so files downloaded from Colab can be dropped straight into models/ and loaded with
# load_checkpoint() there, with no conversion step and no unsafe weights_only=False pickling.

MODEL_REGISTRY = {
    "StackedLSTM": StackedLSTM,
    "ConvLSTM": ConvLSTM,
    "TransformerEncoderModel": TransformerEncoderModel,
    "HARRVLSTMHybrid": HARRVLSTMHybrid,
}

def save_checkpoint_colab(model, checkpoint_path, model_class_name, constructor_kwargs, extra_meta=None):
    torch.save(model.state_dict(), checkpoint_path)
    sidecar = {"model_class": model_class_name, "constructor_kwargs": constructor_kwargs}
    if extra_meta:
        sidecar.update(extra_meta)
    with open(checkpoint_path + ".json", "w") as f:
        json.dump(sidecar, f, indent=2)

def load_checkpoint_colab(checkpoint_path, map_location="cpu"):
    with open(checkpoint_path + ".json", "r") as f:
        sidecar = json.load(f)
    model = MODEL_REGISTRY[sidecar["model_class"]](**sidecar["constructor_kwargs"])
    state_dict = torch.load(checkpoint_path, map_location=map_location, weights_only=True)
    model.load_state_dict(state_dict)
    for key in ("data_contract_version", "lookback_window", "horizons", "tensor_orientation", "train_mean", "train_std"):
        if key in sidecar:
            setattr(model, key, sidecar[key])
    model.to(map_location)
    model.eval()
    return model

def build_model(model_name, horizon):
    """Returns (model, model_class_name, model_kwargs) for the given model_name."""
    if model_name == "convlstm":
        kwargs = dict(in_channels=1, hidden_dims=[32, 64, 64], kernel_size=3, num_layers=3, horizon=horizon)
        return ConvLSTM(**kwargs), "ConvLSTM", kwargs
    elif model_name == "lstm":
        kwargs = dict(grid_size=(7, 7), hidden_dim=256, num_layers=3, horizon=horizon)
        return StackedLSTM(**kwargs), "StackedLSTM", kwargs
    elif model_name == "transformer":
        kwargs = dict(grid_size=(7, 7), horizon=horizon)
        return TransformerEncoderModel(**kwargs), "TransformerEncoderModel", kwargs
    elif model_name == "hybrid":
        kwargs = dict(grid_size=(7, 7), hidden_dim=64, num_layers=3, horizon=horizon)
        return HARRVLSTMHybrid(**kwargs), "HARRVLSTMHybrid", kwargs
    else:
        raise ValueError(f"Unknown model name: {model_name}")

def train_model(model_name, X_train, y_train, X_val, y_val, lookback, horizon, epochs=30, batch_size=32):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Training {model_name} on {device}...")

    model, model_class_name, model_kwargs = build_model(model_name, horizon)

    # This notebook does not standardize surfaces (IV values are already in a narrow, well-scaled
    # band). train_mean/std are recorded as the identity transform (0.0 / 1.0) so the Hybrid model's
    # internal denormalization step, and the local serving code that always applies train_mean/std,
    # stay consistent with "no normalization was actually applied."
    model.data_contract_version = "v1.0"
    model.lookback_window = lookback
    model.horizons = [horizon]
    model.tensor_orientation = "(B, L, C, E, M)"
    model.train_mean = 0.0
    model.train_std = 1.0

    if model_name == "hybrid":
        print("Fitting HAR-RV coefficients on training sequences...")
        model.fit_har(X_train, y_train)

    model = model.to(device)
    criterion = SmoothnessRegularizedLoss(lambda_strike=0.01, lambda_expiry=0.01, grid_taus=GRID_TAUS)
    optimizer = optim.AdamW(model.parameters(), lr=0.001)
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs)

    train_loader = DataLoader(SurfaceDataset(X_train, y_train), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(SurfaceDataset(X_val, y_val), batch_size=batch_size, shuffle=False)

    best_val_loss = float("inf")
    best_path = f"{model_name}_best.pt"

    def _checkpoint_meta():
        return {
            "data_contract_version": model.data_contract_version,
            "lookback_window": model.lookback_window,
            "horizons": model.horizons,
            "tensor_orientation": model.tensor_orientation,
            "train_mean": model.train_mean,
            "train_std": model.train_std,
        }

    for epoch in range(1, epochs + 1):
        model.train()
        train_loss_accum = 0.0
        for X_b, y_b in train_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            optimizer.zero_grad()
            y_p = model(X_b)
            residual = getattr(model, "last_residual", None)
            loss, _ = criterion(y_p, y_b, residual=residual)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_loss_accum += loss.item() * X_b.size(0)

        train_loss = train_loss_accum / len(train_loader.dataset)
        scheduler.step()

        model.eval()
        val_loss_accum = 0.0
        with torch.no_grad():
            for X_b, y_b in val_loader:
                X_b, y_b = X_b.to(device), y_b.to(device)
                y_p = model(X_b)
                residual = getattr(model, "last_residual", None)
                loss, _ = criterion(y_p, y_b, residual=residual)
                val_loss_accum += loss.item() * X_b.size(0)
        val_loss = val_loss_accum / len(val_loader.dataset)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            save_checkpoint_colab(model, best_path, model_class_name, model_kwargs, extra_meta=_checkpoint_meta())

        if epoch % 5 == 0 or epoch == epochs:
            gate_str = f" | Gate={model.gate_value:.4f}" if hasattr(model, "gate_value") else ""
            print(f"Epoch {epoch:02d} | Train Loss={train_loss:.6f} | Val Loss={val_loss:.6f}{gate_str}")

    print(f"Finished training {model_name}. Champion model loaded.")
    return load_checkpoint_colab(best_path, map_location=device)

## Run Benchmarking Experiment

In [ ]:
# 8. Run comparative evaluation

lookback = 60
horizon = 5
epochs = 100

# Generate datasets (seeded for reproducibility across Colab runs)
dataset = generate_synthetic_dataset(num_days=3000, seed=SEED)
T = len(dataset)
train_end = int(T * 0.6)
val_end = int(T * 0.8)

train_data = dataset[:train_end]
val_data = dataset[train_end:val_end]
test_data = dataset[val_end:]

X_train, y_train = create_sequences(train_data, lookback, horizon)
X_val, y_val = create_sequences(val_data, lookback, horizon)
X_test, y_test = create_sequences(test_data, lookback, horizon)

masks = get_region_masks(GRID_KAPPAS, GRID_TAUS)
results = {}

# 1. Random Walk
rw = NaiveRandomWalk(horizon=horizon)
rw_pred = rw.predict(X_test)
results["Random Walk"] = calculate_metrics(rw_pred, y_test)

# 2. Historical Mean
hm = HistoricalMean(horizon=horizon)
hm_pred = hm.predict(X_test)
results["Historical Mean"] = calculate_metrics(hm_pred, y_test)

# 3. Exponential Smoothing
es = ExponentialSmoothing(horizon=horizon)
es_pred = es.predict(X_test)
results["Exp. Smoothing"] = calculate_metrics(es_pred, y_test)

# 4. HAR-RV
har = HARRVModel(horizon=horizon)
har.fit(X_train, y_train)
har_pred = har.predict(X_test)
results["HAR-RV"] = calculate_metrics(har_pred, y_test)

# 5. Stacked LSTM
lstm_model = train_model("lstm", X_train, y_train, X_val, y_val, lookback, horizon, epochs=epochs)
with torch.no_grad():
    device = next(lstm_model.parameters()).device
    lstm_pred = lstm_model(torch.from_numpy(X_test).float().to(device)).cpu().numpy()
results["LSTM"] = calculate_metrics(lstm_pred, y_test)

# 6. ConvLSTM (Smooth)
conv_model = train_model("convlstm", X_train, y_train, X_val, y_val, lookback, horizon, epochs=epochs)
with torch.no_grad():
    device = next(conv_model.parameters()).device
    conv_pred = conv_model(torch.from_numpy(X_test).float().to(device)).cpu().numpy()
results["ConvLSTM (Smooth)"] = calculate_metrics(conv_pred, y_test)

# 7. Transformer Encoder
transformer_model = train_model("transformer", X_train, y_train, X_val, y_val, lookback, horizon, epochs=epochs)
with torch.no_grad():
    device = next(transformer_model.parameters()).device
    transformer_pred = transformer_model(torch.from_numpy(X_test).float().to(device)).cpu().numpy()
results["Transformer"] = calculate_metrics(transformer_pred, y_test)

# 8. HAR-RV + LSTM Hybrid
hybrid_model = train_model("hybrid", X_train, y_train, X_val, y_val, lookback, horizon, epochs=epochs)
with torch.no_grad():
    device = next(hybrid_model.parameters()).device
    hybrid_pred = hybrid_model(torch.from_numpy(X_test).float().to(device)).cpu().numpy()
results["Hybrid (HAR+LSTM)"] = calculate_metrics(hybrid_pred, y_test)

# Export every champion model as state_dict + JSON sidecar for local serving --
# matches models/serialization.py exactly, so these files can be dropped straight into models/.
export_specs = [
    (lstm_model, "checkpoint_lstm.pt"),
    (conv_model, "checkpoint_convlstm.pt"),
    (transformer_model, "checkpoint_transformer.pt"),
    (hybrid_model, "checkpoint_hybrid.pt"),
]
for trained_model, out_path in export_specs:
    model_cpu = trained_model.to("cpu")
    model_class_name = type(model_cpu).__name__
    _, _, model_kwargs = build_model(
        {"StackedLSTM": "lstm", "ConvLSTM": "convlstm",
         "TransformerEncoderModel": "transformer", "HARRVLSTMHybrid": "hybrid"}[model_class_name],
        horizon
    )
    save_checkpoint_colab(model_cpu, out_path, model_class_name, model_kwargs, extra_meta={
        "data_contract_version": model_cpu.data_contract_version,
        "lookback_window": model_cpu.lookback_window,
        "horizons": model_cpu.horizons,
        "tensor_orientation": model_cpu.tensor_orientation,
        "train_mean": model_cpu.train_mean,
        "train_std": model_cpu.train_std,
    })
    print(f"--> Exported {out_path} + {out_path}.json")

print("\nDownload every checkpoint_*.pt AND checkpoint_*.pt.json file and place both into models/ locally.")
print("To update the active served model, also copy whichever pair matches configs/base_config.yaml's")
print("model.name (default: hybrid) to models/checkpoint.pt and models/checkpoint.pt.json.")

## Display Final Benchmark Table

In [ ]:
# 9. Print results
print("\n" + "="*90)
print(f"{'Model':<20} | {'RMSE':<8} | {'MAE':<8} | {'TV (Roughness)':<15} | {'AV (Arbitrage Violations)':<25}")
print("-"*90)
for model_name, metrics in results.items():
    print(f"{model_name:<20} | {metrics['RMSE']:.5f} | {metrics['MAE']:.5f} | {metrics['TV']:.5f}         | {metrics['AV']:.4%}")
print("="*90)